# Семинар 2 - Цветовые пространства, гистограммы изображений, интегральные изображения

***

**Данный семинар содержит домашнее задание - оцениваемые упражнения и вопросы.**

Система оценивания: доля правильно решенных упражений. Максимальный балл, соответственно, 1.

В упражнениях оценивается два аспекта:
1. Код проходит assert'ы (если они есть)
2. Код корректен с точки зрения логики

Вопросы также оцениваются. Ответ на них нужно записывать в соответствующие markdown-ячейки.

Источник используемого аэрофотоснимка: https://sovzond.ru/upload/medialibrary/267/%D0%98%D1%81%D1%85%D0%BE%D0%B4%D0%BD%D1%8B%D0%B9-%D0%B0%D1%8D%D1%80%D0%BE%D1%84%D0%BE%D1%82%D0%BE%D1%81%D0%BD%D0%B8%D0%BC%D0%BE%D0%BA.jpg

In [ ]:
from pathlib import Path

import cv2
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
AERIAL_IMG_PATH = "data/aerial_image.jpg"
SUNFLOWER_IMG_PATH = "data/sunflower.jpg"

if not Path(AERIAL_IMG_PATH).exists() or not Path(SUNFLOWER_IMG_PATH).exists():
    !git clone https://github.com/alexmelekhin/cv_course_2023.git
    !mv cv_course_2023/seminars/seminar_02/data .

# 1. Цветовые пространства

Наиболее распространенным представлением цвета пикселя является пространство RGB. В таком представлении цвет представлен тремя числами: интенсивностями красного, зеленого и синего базисных цветов.

In [ ]:
img = cv2.imread(SUNFLOWER_IMG_PATH)

In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
plt.figure(figsize=[3, 3])
plt.imshow(img_rgb);

In [ ]:
plt.figure(figsize=[9, 3])

plt.subplot(131)
plt.imshow(img_rgb[:,:,0], cmap='Reds')
plt.title('Red Channel')

plt.subplot(132)
plt.imshow(img_rgb[:,:,1], cmap='Greens')
plt.title('Green Channel')

plt.subplot(133)
plt.imshow(img_rgb[:,:,2], cmap='Blues')
plt.title('Blue Channel')

plt.show()

cvtColor поддерживает конвертацию между множеством других цветовых схем. К примеру, чтобы получить серое изобаржение из цветного достаточно:

In [ ]:
img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

In [ ]:
plt.figure(figsize=[3, 3])
plt.imshow(img_gray, cmap='Greys_r')
plt.colorbar()
plt.show()

## Вопрос 1

**Этот и все последующие вопросы - оцениваемые, для самостоятельной работы.**

Можно ли получить черно-белое изображение из RGB представления путем вычисления среднего значения интенсивностей каналов R, G и B? Почему?

**Ответ:**

## Вопрос 2

Почему мы использовали флаг `cmap='Greys_r'` при отображении черно-белого изображения? Чем отличается от `cmap='Greys'`?

**Ответ:**

Если не предполагается использовать информацию о цвете пикселей, то можно сразу загрузить изображение в оттенках серого:

In [ ]:
img_gray = cv2.imread(SUNFLOWER_IMG_PATH, cv2.IMREAD_GRAYSCALE)

In [ ]:
plt.figure(figsize=[3, 3])
plt.imshow(img_gray, cmap='Greys_r')
plt.colorbar()
plt.show()

серое изображение - двумерный массив:

In [ ]:
print('type(img_gray) = ', type(img_gray))
print('img_gray.shape = ', img_gray.shape)
print('img_gray.dtype = ', img_gray.dtype)

## Упражнение 1: RGB to gray

**Это и все последующие упражения - оцениваемые, для самостоятельной работы.**

Реализуйте функцию преобразования цветного изображения в формате RGB в серое

In [ ]:
def convert_rgb_to_grayscale(img_rgb):
    r = img_rgb[:, :, 0].astype(np.int64)
    g = img_rgb[:, :, 1].astype(np.int64)
    b = img_rgb[:, :, 2].astype(np.int64)
    luminance = r * 9798 + g * 19235 + b * 3736
    return ((luminance + (1 << 14)) >> 15).astype(np.uint8)


In [ ]:
assert((cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY) == convert_rgb_to_grayscale(img_rgb)).all())

Кроме RGB/BGR и grayscale существуют и другие цветовые представления. Преобразования между ними можно осуществлять с помощью библиотеки OpenCV следующим способом:

## RGB to HSV

HSV - цветовое пространство, в котором цвет представлен тремя компонентами: Hue (оттенок), Saturation (насыщенность) и Value (значение). Это позволяет задавать цвета в более естественной форме, чем RGB.

In [ ]:
img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

можно попробовать отобразить изображение так же, как и RGB:

In [ ]:
plt.figure(figsize=[3, 3])
plt.imshow(img_hsv);

Однако это не совсем корректно, так как Hue - это угол, а не интенсивность. Поэтому лучше отобразить оттенок в виде цвета, а насыщенность и значение - в виде яркости:

In [ ]:
h, s, v = cv2.split(img_hsv)

fig, axs = plt.subplots(1, 3, figsize=[15, 5])

im1 = axs[0].imshow(h, cmap="hsv")
axs[0].set_title('Hue')
cbar1 = plt.colorbar(im1, ax=axs[0], fraction=0.046, pad=0.04, ticks=[0, 90, 179])
cbar1.ax.set_yticklabels(['0', '90', '179'])

im2 = axs[1].imshow(s, cmap="viridis")
axs[1].set_title('Saturation')
cbar2 = plt.colorbar(im2, ax=axs[1], fraction=0.046, pad=0.04, ticks=[0, 128, 255])
cbar2.ax.set_yticklabels(['0', '128', '255'])

im3 = axs[2].imshow(v, cmap="inferno")
axs[2].set_title('Value')
cbar3 = plt.colorbar(im3, ax=axs[2], fraction=0.046, pad=0.04, ticks=[0, 128, 255])
cbar3.ax.set_yticklabels(['0', '128', '255'])

plt.show()

## Упражнение 2

Попробуйте другие цветовые пространства, конвертация в которые реализована в OpenCV.

In [ ]:
img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
img_ycrcb = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2YCrCb)

fig, axs = plt.subplots(2, 3, figsize=[12, 8])

axs[0, 0].imshow(img_rgb)
axs[0, 0].set_title('RGB')

axs[0, 1].imshow(img_lab[:, :, 0], cmap='gray')
axs[0, 1].set_title('LAB: L channel')

axs[0, 2].imshow(img_lab[:, :, 1], cmap='PiYG')
axs[0, 2].set_title('LAB: a channel')

axs[1, 0].imshow(cv2.cvtColor(img_ycrcb, cv2.COLOR_YCrCb2RGB))
axs[1, 0].set_title('YCrCb reconstructed')

axs[1, 1].imshow(img_ycrcb[:, :, 1], cmap='Reds')
axs[1, 1].set_title('YCrCb: Cr channel')

axs[1, 2].imshow(img_ycrcb[:, :, 2], cmap='Blues')
axs[1, 2].set_title('YCrCb: Cb channel')

for ax in axs.ravel():
    ax.axis('off')

plt.tight_layout()
plt.show()


## Вопрос 3

В каких задачах переход из RGB в другое цветовое пространство может быть полезным?

**Ответ:**

# 2. Гистограмма изображения

Напомним, что гистограммой изображения называется функция, показыващая количество пикселей изображения с заданным в качестве аргумента значением интенсивности:

$$
    h(v) = \sum_{x = 0}^{W - 1} \sum_{y = 0}^{H - 1} [f(x, y) = v] 
$$

Если дополнительно потребовать, чтобы $\sum_{v = 0}^{255} h(v) = 1$, то $h$ будет представлять функцию плотности распределения интенсивности на изображении.

Определим вспомогательную функцию, позволяющую визуализировать гистограмму:

In [ ]:
def visualize_hist(hist):
    plt.figure(figsize=[12, 3])
    plt.bar(np.arange(len(hist)), hist / hist.sum())

Рассчитать гистограмму можно с помощью встроенной функции OpenCV:

In [ ]:
hist_cv = cv2.calcHist([img_gray],      # список изображений
                       [0],
                       None,
                       [256],
                       [0, 256])[:, 0]  # диапазон значений

Результат представляет собой обычный массив длины 256:

In [ ]:
print('hist_cv.shape = ', hist_cv.shape)

и выглядит следующим образом:

In [ ]:
visualize_hist(hist_cv)

## Упражнение 3: Построение гистограммы

Реализуйте функцию для расчета гистограммы изображения. Используйте ее для визуализации трех каналов RGB изображения.

In [ ]:
def compute_hist(img):
    flat = img.ravel()
    hist = np.zeros(256, dtype=np.float32)
    unique, counts = np.unique(flat, return_counts=True)
    hist[unique] = counts.astype(np.float32)
    return hist


In [ ]:
hist_gray = compute_hist(img_gray)

hist_r = compute_hist(img_rgb[:, :, 0])
hist_g = compute_hist(img_rgb[:, :, 1])
hist_b = compute_hist(img_rgb[:, :, 2])

In [ ]:
visualize_hist(hist_r)

In [ ]:
visualize_hist(hist_g)

In [ ]:
visualize_hist(hist_b)

In [ ]:
assert((hist_gray == hist_cv).all())

## Вопрос 4

 Что можно сказать об изображении по его гистограмме?

**Ответ:**

## Вопрос 5

Допустим, вы смотрите некоторый фильм и для текущего кадра выводите его гистограмму. Как будет меняться эта гистограмма с течением времени? Опишите несколько случаев: смена камеры, смена освещения, смена сцены.

**Ответ:**

## Упражнение 4: JPEG и гистограмма

Исследуйте, как влияет степень сжатия алгоритма JPEG на вид гистограммы изображения.

Используйте черно-белое изображение `img_gray`. Визуализируйте степени сжатия 90, 60, 30, 5.

In [ ]:
qualities = [90, 60, 30, 5]

fig, axs = plt.subplots(2, len(qualities), figsize=[4 * len(qualities), 6])

for i, quality in enumerate(qualities):
    ok, encoded = cv2.imencode('.jpg', img_gray, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    assert ok

    img_jpeg = cv2.imdecode(encoded, cv2.IMREAD_GRAYSCALE)
    hist_jpeg = compute_hist(img_jpeg)

    axs[0, i].imshow(img_jpeg, cmap='Greys_r')
    axs[0, i].set_title(f'JPEG quality = {quality}')
    axs[0, i].axis('off')

    axs[1, i].bar(np.arange(256), hist_jpeg / hist_jpeg.sum(), width=1.0)
    axs[1, i].set_xlim(0, 255)
    axs[1, i].set_title('Histogram')

plt.tight_layout()
plt.show()


## Упражнение 5: Сегментация

На загруженном аэроортофотоплане выделите зеленые насаждения. Для этого постройте бинарную маску, где 1 будет отвечать наличию насаждений в данном пикселе, 0 - их отсутствию, и визуализируйте её. А также рассчитайте, какую долю изображения занимают зеленые насаждения. С какой ошибкой (погрешностью) получена эта величина?

**Подсказка:** вам должно помочь HSV пространство и гистограмма. Погрешность может быть оценена на глаз, по вашей неуверенности в определении порога отделения классов 'зеленые насаждения'/'прочее'.

In [ ]:
aerial_image = cv2.cvtColor(cv2.imread(AERIAL_IMG_PATH), cv2.COLOR_BGR2RGB)

plt.figure(figsize=[10, 10])
plt.imshow(aerial_image);

In [ ]:
aerial_hsv = cv2.cvtColor(aerial_image, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(aerial_hsv)

h_min, h_max = 35, 85
s_min, v_min = 40, 35

green_mask = ((h >= h_min) & (h <= h_max) & (s >= s_min) & (v >= v_min)).astype(np.uint8)
kernel = np.ones((5, 5), np.uint8)
green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_OPEN, kernel)
green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_CLOSE, kernel)

green_share = green_mask.mean()

shares = []
for h_lo in [33, 35, 37]:
    for h_hi in [83, 85, 87]:
        for s_thr in [35, 40, 45]:
            m = ((h >= h_lo) & (h <= h_hi) & (s >= s_thr) & (v >= v_min)).astype(np.uint8)
            m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel)
            shares.append(m.mean())
error_abs = (max(shares) - min(shares)) / 2

overlay = aerial_image.copy()
overlay[green_mask == 0] = (overlay[green_mask == 0] * 0.35).astype(np.uint8)

fig, axs = plt.subplots(1, 3, figsize=[18, 6])
axs[0].imshow(aerial_image)
axs[0].set_title('Original image')
axs[0].axis('off')

axs[1].imshow(green_mask, cmap='gray')
axs[1].set_title('Greenery mask')
axs[1].axis('off')

axs[2].imshow(overlay)
axs[2].set_title('Overlay')
axs[2].axis('off')

plt.tight_layout()
plt.show()

print(f'Greenery share: {green_share:.2%}')
print(f'Estimated error: +/- {error_abs:.2%}')


# 3. Интегральные изображения

## Упражнение 6

Напомним, что интегральным изображением называется следующая функция:

$$
    I(x, y) = \sum_{i = 0}^{x} \sum_{j = 0}^{y} f(i, j)
$$

С помощью интегрального изображения можно за $O(1)$ вычислять сумму интенсивностей в произвольной прямоугольной области. Требуется реализовать расчет интегрального изображения, а также быстрый расчет сумм интенсивностей в прямоугольнике заданном верхним левым углом, шириной и высотой $x, y, w, h$.

In [ ]:
class IntegralImage:

    def __init__(self, img):
        assert(len(img.shape) == 2)

        self.img = img
        self.integral = img.astype(np.int64).cumsum(axis=0).cumsum(axis=1)

    def sum(self, x, y, w, h):
        x2, y2 = x + w - 1, y + h - 1
        total = int(self.integral[y2, x2])

        if x > 0:
            total -= int(self.integral[y2, x - 1])
        if y > 0:
            total -= int(self.integral[y - 1, x2])
        if x > 0 and y > 0:
            total += int(self.integral[y - 1, x - 1])

        return total


In [ ]:
I = IntegralImage(img_gray)

In [ ]:
x, y, w, h = 0, 0, 100, 100
assert(img_gray[y:y + h, x:x + w].sum() == I.sum(x, y, w, h))

x, y, w, h = 100, 100, 100, 100
assert(img_gray[y:y + h, x:x + w].sum() == I.sum(x, y, w, h))

## Вопрос 6

В каких задачах может потребоваться использовать интегральное изображение?

**Ответ:**

## Вопрос 7

Какому методу решения задачи в программировании следует метод расчета интегрального изображения?

**Ответ:**